In [2]:
import os
import json
import shutil
from google.colab import drive

# CONFIGURATION: Set your Drive Paths
# Replace with your actual unzipped folder path in Drive
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/coco2017/train2017"
DRIVE_ANNOTATIONS = "/content/drive/MyDrive/coco2017/annotations/captions_train2017.json"

# Local fast storage on Colab
LOCAL_TRAIN_DIR = "/content/lisa_training_data"

def prepare_colab_training_subset(sample_limit=5000):
    """
    Mounts Drive and moves a subset of unzipped images to local disk
    to ensure training speed.
    """
    # 1. Mount Drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # 2. Create local directory
    if not os.path.exists(LOCAL_TRAIN_DIR):
        os.makedirs(LOCAL_TRAIN_DIR)

    # 3. Load Annotations
    print("Reading COCO Annotations from Drive...")
    with open(DRIVE_ANNOTATIONS, 'r') as f:
        coco_data = json.load(f)

    id_to_file = {img['id']: img['file_name'] for img in coco_data['images']}

    # 4. Create Metadata Linker & Copy Images
    print(f"Linking and copying {sample_limit} images to local SSD...")
    metadata_path = os.path.join(LOCAL_TRAIN_DIR, "metadata.jsonl")

    count = 0
    with open(metadata_path, 'w') as f_meta:
        for ann in coco_data['annotations']:
            if count >= sample_limit:
                break

            img_id = ann['image_id']
            filename = id_to_file.get(img_id)

            if filename:
                source_path = os.path.join(DRIVE_IMAGES_DIR, filename)
                target_path = os.path.join(LOCAL_TRAIN_DIR, filename)

                # Copy the file to local storage if it exists
                if os.path.exists(source_path):
                    if not os.path.exists(target_path):
                        shutil.copy(source_path, target_path)

                    # Write to the linker file
                    line = {"file_name": filename, "text": ann['caption']}
                    f_meta.write(json.dumps(line) + "\n")
                    count += 1

    print(f"Success! {count} images and metadata.jsonl ready in {LOCAL_TRAIN_DIR}")

if __name__ == "__main__":
    prepare_colab_training_subset(sample_limit=5000)

Mounted at /content/drive
Reading COCO Annotations from Drive...
Linking and copying 5000 images to local SSD...
Success! 5000 images and metadata.jsonl ready in /content/lisa_training_data


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import subprocess
import sys

# 1. ENVIRONMENT SETUP

def install_requirements():
    print("Installing training dependencies...")
    # Using specific versions for stability on Colab
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "peft", "bitsandbytes"])

    if not os.path.exists("train_text_to_image_lora.py"):
        print("Fetching official training script...")
        subprocess.check_call(["wget", "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py"])

# 2. TRAINING EXECUTION

def launch_lora_training(data_dir="/content/lisa_training_data"):
    print("STARTING LORA TRAINING (500 Steps)...")
    print("This will take approximately 30-60 minutes on a T4 GPU.")

    # We save output to Drive so we don't lose it
    output_dir = "/content/drive/MyDrive/lisa_model_output"

    # This command triggers the actual heavy lifting
    !accelerate launch --mixed_precision="fp16" train_text_to_image_lora.py \
      --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
      --train_data_dir="{data_dir}" \
      --caption_column="text" \
      --resolution=512 \
      --train_batch_size=1 \
      --gradient_accumulation_steps=4 \
      --max_train_steps=500 \
      --learning_rate=1e-04 \
      --lr_scheduler="cosine" \
      --output_dir="{output_dir}" \
      --checkpointing_steps=250 \
      --seed=42

if __name__ == "__main__":
    # First, install everything
    install_requirements()

    # NOW, start the training (Uncommented and ready)
    if os.path.exists("/content/lisa_training_data/metadata.jsonl"):
        launch_lora_training()
    else:
        print("ERROR: Training data not found. Please run the 'Data Bridge' script (Stage 1) first!")

Installing training dependencies...
Fetching official training script...
STARTING LORA TRAINING (500 Steps)...
This will take approximately 30-60 minutes on a T4 GPU.
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-01-06 17:04:35.075188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767719075.108248   11898 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767719075.120086   11898 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register 

In [5]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from openai import OpenAI
import os
from IPython.display import display, clear_output
from PIL import Image
import ipywidgets as widgets
from transformers import CLIPProcessor, CLIPModel


# 1. EVALUATION SUITE

class LISAEvaluator:
    """
    Handles the quantitative and qualitative evaluation of the pipeline.
    """
    def __init__(self, device="cuda"):
        print("Initializing Evaluation Suite (CLIP)...")
        self.device = device
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def calculate_clip_score(self, image, text):
        """Calculates the alignment score between the image and the prompt."""
        inputs = self.clip_processor(text=[text], images=image, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.clip_model(**inputs)

        # Cosine similarity score (normalized to 0-100 range)
        logits_per_image = outputs.logits_per_image
        score = logits_per_image.item()
        return round(score, 2)

    def judge_reasoning(self, client, original, expanded):
        """Uses LLM-as-a-judge to evaluate the LISA expansion quality."""
        prompt = (
            f"Compare the original intent with the expanded LISA prompt.\n"
            f"Original: {original}\nExpanded: {expanded}\n"
            f"Evaluate if the expansion adds useful COCO-style spatial and material details. "
            f"Rate from 1-10 and give a 1-sentence justification."
        )
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except:
            return "Evaluation unavailable."





Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [6]:
# 2. CORE PIPELINE CLASS (OpenAI Version)

class LISAProjectOpenAI:
    def __init__(self, openai_key, hf_token, lora_path):
        self.lora_path = lora_path
        self.client = OpenAI(api_key=openai_key)

        # Load Base Model on T4 GPU
        print("Loading Stable Diffusion Base Model...")
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-v1-5",
            torch_dtype=torch.float16,
            use_auth_token=hf_token
        ).to("cuda")

        # Initialize Evaluator
        self.evaluator = LISAEvaluator(device="cuda")

        # Load LoRA Weights
        self.has_lora = False
        if os.path.exists(self.lora_path):
            if os.path.isdir(self.lora_path):
                if "pytorch_lora_weights.safetensors" in os.listdir(self.lora_path):
                    self.has_lora = True
            else:
                self.has_lora = True

        if self.has_lora:
            self.pipe.load_lora_weights(self.lora_path)
            print(f"SUCCESSFULLY LOADED LOCAL COCO WEIGHTS.")
        else:
            print(f"WARNING: Weights not found at {self.lora_path}. Running base model.")

        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(self.pipe.scheduler.config)
        print("Pipeline Initialized.")

    def generate_result(self, user_intent, output_widget, use_lora=True):
        with output_widget:
            print(f"\nSTEP 1: LISA (OpenAI) is performing Semantic Expansion...")

            system_instruction = (
                "You are a LISA reasoning agent. Your task is to expand user prompts into "
                "high-detail descriptions focused on spatial relations (next to, on top of, inside) "
                "grounded in the COCO dataset semantics. "
                "IMPORTANT: Output your response exactly in this format:\n"
                "PROMPT: [your expanded prompt] | NEGATIVE: [things to avoid]"
            )

            try:
                response = self.client.chat.completions.create(
                    model="gpt-4",
                    messages=[
                        {"role": "system", "content": system_instruction},
                        {"role": "user", "content": user_intent}
                    ]
                )
                content = response.choices[0].message.content
                expanded_prompt = content.split("PROMPT:")[1].split("|")[0].strip()
                negative_prompt = content.split("NEGATIVE:")[1].strip()
            except Exception as e:
                print(f"OpenAI processing failed. Using fallback.")
                expanded_prompt = f"{user_intent}, high quality, highly detailed, COCO style"
                negative_prompt = "blurry, low quality, distorted"

            print(f" LISA Expanded Prompt: {expanded_prompt}")

            # Toggle LoRA for evaluation comparison if requested
            if not use_lora and self.has_lora:
                self.pipe.unload_lora_weights()
                print("Temporarily unloading LoRA for baseline comparison...")

            print(f"\n STEP 2: Rendering pixels...")
            image = self.pipe(
                prompt=expanded_prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=30,
                guidance_scale=8.0
            ).images[0]

            # Reload LoRA if we unloaded it
            if not use_lora and self.has_lora:
                self.pipe.load_lora_weights(self.lora_path)

            # STEP 3: EVALUATION
            print("\nSTEP 3: Running Evaluation Metrics...")
            clip_score = self.evaluator.calculate_clip_score(image, expanded_prompt)
            reasoning_score = self.evaluator.judge_reasoning(self.client, user_intent, expanded_prompt)

            return image, expanded_prompt, clip_score, reasoning_score

In [7]:
# 3. INTERACTIVE DASHBOARD UI

def create_dashboard():
    style = {'description_width': 'initial'}
    header = widgets.HTML("<h2>LISA: Reasoning-Based Image Synthesis & Evaluation</h2>")

    # API Inputs
    openai_input = widgets.Password(description="OpenAI API Key:", style=style)
    hf_input = widgets.Password(description="HF Token:", style=style)
    lora_input = widgets.Text(description="LoRA Path:", value="/content/drive/MyDrive/lisa_model_output", style=style)

    # Evaluation Toggle
    eval_mode = widgets.Dropdown(
        options=[('Trained (LoRA)', True), ('Baseline (Base Model)', False)],
        value=True,
        description='Model Mode:',
        style=style
    )

    prompt_input = widgets.Textarea(description="User Intent:", placeholder="e.g., A red bowl on a wooden table next to a window", layout=widgets.Layout(width='95%', height='80px'))
    generate_btn = widgets.Button(description="Generate & Evaluate", button_style='primary', layout=widgets.Layout(width='95%'))

    status_output = widgets.Output()
    image_output = widgets.Output()
    metrics_output = widgets.Output()

    def on_click(b):
        with status_output:
            clear_output()
            if not openai_input.value or not hf_input.value:
                print("Error: Missing API Keys or Tokens.")
                return

            try:
                if not hasattr(on_click, "project"):
                    on_click.project = LISAProjectOpenAI(openai_input.value, hf_input.value, lora_input.value)

                img, exp_p, clip, reason = on_click.project.generate_result(prompt_input.value, status_output, use_lora=eval_mode.value)

                with image_output:
                    clear_output()
                    display(img)

                with metrics_output:
                    clear_output()
                    display(widgets.HTML(f"""
                    <div style="border: 1px solid #4F46E5; padding: 15px; border-radius: 10px; background-color: #F5F3FF;">
                        <h4 style="margin-top:0; color:#4F46E5;">📊 Evaluation Results</h4>
                        <p><b>CLIP Score (Alignment):</b> <span style="font-size: 1.2em; color: green;">{clip}</span></p>
                        <p><b>Reasoning Critique:</b> {reason}</p>
                        <p><small>*CLIP scores above 25 indicate strong text-image alignment.</small></p>
                    </div>
                    """))
            except Exception as e:
                print(f"Pipeline Error: {e}")

    generate_btn.on_click(on_click)

    # Layout
    main_ui = widgets.VBox([
        header,
        widgets.VBox([openai_input, hf_input, lora_input, eval_mode], layout=widgets.Layout(padding='10px', border='1px solid #ddd')),
        widgets.HTML("<b>Generation & Evaluation</b>"),
        prompt_input, generate_btn,
        widgets.HBox([
            widgets.VBox([widgets.HTML("<b>Logs</b>"), status_output], layout=widgets.Layout(width='50%')),
            widgets.VBox([widgets.HTML("<b>Metrics</b>"), metrics_output], layout=widgets.Layout(width='50%'))
        ]),
        widgets.HTML("<b>Final Output</b>"),
        image_output
    ], layout=widgets.Layout(width='100%', padding='20px'))

    display(main_ui)

if __name__ == "__main__":
    if torch.cuda.is_available():
        create_dashboard()
    else:
        print("ERROR: T4 GPU not detected.")